In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import Adam
from keras import backend as K



In [2]:

train_dir = 'C:\\Users\\arunk\\OneDrive\\Documents\\GenAI - Scaler\\Classification\\archive\\data3a\\training'
test_dir = 'C:\\Users\\arunk\\OneDrive\\Documents\\GenAI - Scaler\\Classification\\archive\\data3a\\validation'


In [3]:
img_height = 224
img_width = 224
no_train_img = 1400
no_test_img = 500
epochs = 10
batch_size = 32

In [4]:
if K.image_data_format() == 'channels first':
    input_shape = (3,img_width,img_height)
else:
    input_shape = (img_width,img_height,3)

In [5]:
train_datagen = ImageDataGenerator(
    rescale = 1. / 255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip = True)

test_datagen = ImageDataGenerator(
    rescale = 1. / 255)

In [6]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width,img_height),
    batch_size = batch_size,
    class_mode = 'categorical')

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_width,img_height),
    batch_size = batch_size,
    class_mode = 'categorical')


Found 1383 images belonging to 3 classes.
Found 248 images belonging to 3 classes.


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Activation # type: ignore

model = Sequential()

model.add(Conv2D(32,(3,3),input_shape = (img_width,img_height,3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size = (2,2)))

model.add(Conv2D(64,(3,3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size = (2,2)))

#increae the extraction of features from 32 to 64

model.add(Conv2D(64,(3,3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size = (2,2)))


#flatten the model to get single Input Layer
model.add(Flatten())                                #image as 1D array
model.add(Dense(128))                                #give 64 features as 64 Nodes
model.add(Activation('relu'))
#model.add(Dropout(0.5))                             
model.add(Dense(3))                                 #3 classes
model.add(Activation('softmax'))

model.summary()

C:\Users\arunk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 222, 222, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 109, 109, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 52, 52, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 43264)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     5,537,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 3)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,594,627 (21.34 MB)

 Trainable params: 5,594,627 (21.34 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(loss = 'categorical_crossentropy', optimizer=Adam(learning_rate=0.0001),metrics = ['accuracy'])

In [9]:
model.fit(
        train_generator,
        validation_data=test_generator,
        epochs=epochs,
    )

Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 102s 2s/step - accuracy: 0.4179 - loss: 1.0764 - val_accuracy: 0.5121 - val_loss: 0.9925
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 81s 2s/step - accuracy: 0.4982 - loss: 0.9946 - val_accuracy: 0.5524 - val_loss: 0.9385
Epoch 3/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 76s 2s/step - accuracy: 0.5206 - loss: 0.9415 - val_accuracy: 0.5685 - val_loss: 0.8986
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 75s 2s/step - accuracy: 0.5278 - loss: 0.9349 - val_accuracy: 0.5887 - val_loss: 0.8930
Epoch 5/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.5445 - loss: 0.9298 - val_accuracy: 0.4960 - val_loss: 0.9916
Epoch 6/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.5900 - loss: 0.8934 - val_accuracy: 0.5887 - val_loss: 0.8802
Epoch 7/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 90s 2s/step - accuracy: 0.5821 - loss: 0.8744 - val_accuracy: 0.5766 - val_loss: 0.8820
Epoch 8/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 0.5980 - loss: 0.8746 - val_accuracy: 0.5927 - val_loss

In [10]:
img_pred = image.load_img(r'C:\Users\arunk\OneDrive\Documents\GenAI - Scaler\Classification\archive\data3a\validation\01-minor\0003.JPEG',target_size=(224,224))
img_pred = image.img_to_array(img_pred)
img_pred = np.expand_dims(img_pred,axis=0)

result = model.predict(img_pred)

result

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 378ms/step


array([[0.0000000e+00, 2.8758331e-09, 1.0000000e+00]], dtype=float32)